In [1]:
from flask import Flask, render_template, request, redirect, url_for, session, flash,jsonify
import mysql.connector
import os
from werkzeug.security import generate_password_hash, check_password_hash
from datetime import datetime
from flask import Flask, request, jsonify
from flask import Flask, render_template, request, redirect, url_for, session, flash,jsonify
import time
import os

from datetime import datetime

app = Flask(__name__)
app.secret_key = "your_secret_key"

In [2]:
import mysql.connector
from mysql.connector import Error

db_config = {
    'host': 'localhost',
    'user': 'root',        
    'password': 'NewPassword123!', 
    'database': 'product_tracking'  ,
     'charset':"utf8"
}

def get_connection():
    try:
        conn = mysql.connector.connect(**db_config)
        return conn
    except Error as e:
        print("DB Connection Error:", e)
        return None

In [3]:
get_connection()

In [4]:

import os
from pathlib import Path
from web3 import Web3
import json


url = "http://localhost:7545"
web3 = Web3(Web3.HTTPProvider(url))
print(web3.isConnected())
web3.eth.defaultAccount = web3.eth.accounts[0]
address=web3.toChecksumAddress('0x0C83Af2f6B70F1E87FD25dD39cd43241600b6307')
abi = json.loads('[{"constant":true,"inputs":[],"name":"get_data","outputs":[{"name":"","type":"string"}],"payable":false,"stateMutability":"view","type":"function"},{"constant":false,"inputs":[{"name":"_name","type":"string"}],"name":"set_data","outputs":[],"payable":false,"stateMutability":"nonpayable","type":"function"},{"constant":true,"inputs":[],"name":"data","outputs":[{"name":"","type":"string"}],"payable":false,"stateMutability":"view","type":"function"}]')
contract = web3.eth.contract(address=address, abi=abi)


address1=web3.toChecksumAddress('0xF9A78ae4153eA9038eC1DA386FA57b46121DBfB6')
abi1 = json.loads('[{"constant":true,"inputs":[],"name":"get_data","outputs":[{"name":"","type":"string"}],"payable":false,"stateMutability":"view","type":"function"},{"constant":false,"inputs":[{"name":"_name","type":"string"}],"name":"set_data","outputs":[],"payable":false,"stateMutability":"nonpayable","type":"function"},{"constant":true,"inputs":[],"name":"data","outputs":[{"name":"","type":"string"}],"payable":false,"stateMutability":"view","type":"function"}]')
contract1 = web3.eth.contract(address=address1, abi=abi1)


True


In [5]:
def transfer_ethers(from_ac,to_ac,pkey,amt):
    account_1 = from_ac
    account_2 = to_ac
    private_key = pkey
    nonce = web3.eth.getTransactionCount(account_1)
    tx = {
    'nonce': nonce,
    'to': account_2,
    'value': web3.toWei(amt, 'ether'),
    'gas': 2000000,
    'gasPrice': web3.toWei('2', 'gwei'),
    }
    signed_tx = web3.eth.account.signTransaction(tx, private_key)

    tx_hash = web3.eth.sendRawTransaction(signed_tx.rawTransaction)

    print(web3.toHex(tx_hash))

In [6]:
import smtplib
from email.message import EmailMessage

def send_email_with_attachment(sender_email, sender_password, recipient_email, subject, body, attachment_path):
    try:
        # Create the email message
        msg = EmailMessage()
        msg['From'] = sender_email
        msg['To'] = recipient_email
        msg['Subject'] = subject
        msg.set_content(body)

        # Attach the file
        with open(attachment_path, 'rb') as f:
            file_data = f.read()
            file_name = f.name.split('/')[-1]  # extract only the filename
        msg.add_attachment(file_data, maintype='application', subtype='octet-stream', filename=file_name)

        # Connect to Gmail's SMTP server
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(sender_email, sender_password)
            smtp.send_message(msg)

        print("Email sent successfully!")

    except Exception as e:
        print("Failed to send email:", e)





In [7]:
import csv
from datetime import datetime

@app.route('/transport', methods=['POST'])
def transport():
    try:
        data = request.get_json()
        print("Received data:", data)

        da = data.get("data")
        print('da',da)
        f=open('data.txt','w')
        f.write(da)
        f.close()
        l = da.split('#')

        temp = l[0]
        humi = l[1]
        mq = l[2]
        lat = l[3]
        lon = l[4]
        print(l)

        
        filename = "data_log.csv"
        now = datetime.now()
        date_str = now.strftime("%Y-%m-%d")
        time_str = now.strftime("%H:%M:%S")

        # If file is empty, write header once
        try:
            with open(filename, 'x', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["Date", "Time", "Temperature", "Humidity", "MQ", "Latitude", "Longitude"])
        except FileExistsError:
            pass  # File already exists

        # Append new data row
        with open(filename, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([date_str, time_str, temp, humi, mq, lat, lon])

        return jsonify({"status": "success"}), 200

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400


In [8]:
@app.route('/register', methods=['POST'])
def register_user():
    fname = request.form.get('fname')
    lname = request.form.get('lname')
    email = request.form.get('email')
    mobile = request.form.get('phone')
    password = request.form.get('pass')
    name=fname+' '+lname
    print(name,email,mobile,password)
    if not (name and email and password):
        return "All fields are required", 400

    conn = get_connection()
    if not conn:
        return "Database connection failed", 500

    cursor = conn.cursor()
    

    try:
        # Check if email exists
        cursor.execute("SELECT * FROM user_details WHERE email = %s", (email,))
        existing = cursor.fetchone()
        if existing:
            return "Email already registered", 409

        # Insert new user
        cursor.execute(
            "INSERT INTO user_details (name, email,mobile, password) VALUES (%s, %s, %s, %s)",
            (name, email,mobile, password)
        )
        conn.commit()
        return "Registration Successful"
    except Error as e:
        print("MySQL Error:", e)
        return "Registration Failed"
    finally:
        cursor.close()
        conn.close()

@app.route('/loginm', methods=['POST'])
def loginm():
    email = request.form.get('email')
    password = request.form.get('password')

    if not email or not password:
        return "Missing email or password", 400

    conn = get_connection()
    if not conn:
        return "Database connection failed", 500

    cursor = conn.cursor(dictionary=True)
    try:
        cursor.execute("SELECT * FROM user_details WHERE email=%s AND password=%s", (email, password))
        user = cursor.fetchone()
        if user:
            return jsonify({
                "status": "success",
                "message": "Login Successful",
                "name": user["name"],
                "email": user["email"],
                "mobile": user["mobile"]
            })
        else:
            return jsonify({
                "status": "fail",
                "message": "Invalid credentials"
            }), 401
    except Error as e:
        print("Login Error:", e)
        return "Login Failed", 500
    finally:
        cursor.close()
        conn.close()


In [9]:
@app.route('/get_qr_details', methods=['POST'])
def get_qr_details():
    qr = request.form.get('qr')
    print('qr',qr)
    lis=qr.split('#')
    '''
    f=open('product_id.txt','w')
    f.write(str(l[0]))
    f.close()
    '''
    product_id=lis[0]
    chain_data=contract.functions.get_data().call()
    print("old data",chain_data)
    l=chain_data.split('%%@@')
    print('l',l)
    ll=[]
    for i in range(len(l)):
        if len(l[i])!=0:
            ll.append(l[i])
    print('ll',ll)
    details=''
    
    message='Not Matched'
    
    for x in ll:
        print('x',x)
        if len(x)>8:
            l1=x.split('#')
            print('l1',l1)
            l2=[]
            for i in range(len(l1)):
                if len(l1[i])!=0:
                    l2.append(l1[i])
            print(l2)
            print(l2[0],product_id)

            if str(l2[0])==product_id:
                details=x
                print('matched',x)
                message='Matched'
                break
     
    
    79#Grapes#Fruits#60.0#B888#2025-11-26#Fresh#1764141267416_download (3).jpg#maruti#0#
    return jsonify({
        "status": "success",
        "message": message,
        "pid": lis[0],
        "pname": lis[1],
        "category": lis[2],
        "price": lis[3],
        "batch": lis[4],
        "date": lis[5],
        "description": lis[6],
        "fname": lis[7],
        "farmer": lis[8]
    })
    
        
    

In [10]:
@app.route('/')
def home():
    if 'email' not in session:
        return redirect(url_for('login'))
    f=open('data.txt','r')
    da=f.read()
    f.close()
    l=da.split('#')
    sensor_data = {
        'temperature': l[0],
        'humidity': l[1],
        'mq': l[2],
        'lat':l[3],
        'lon':l[4]
        
        
    }
    
    #pred = predict([float(l[0]),float(l[1]),float(l[0])])
    
    return render_template('home.html', sensor_data=sensor_data)

In [11]:
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        email = request.form['email']
        password = request.form['password']

        db = get_connection()
        cur = db.cursor(dictionary=True)
        cur.execute("SELECT * FROM user WHERE email=%s", (email,))
        user = cur.fetchone()
        cur.close()
        print('user',user)
        print(user['password'], password)
        if user and user['password']== password:
            print()
            session['email'] = user['email']
            flash("Login successful!", "success")
            return redirect(url_for('home'))
        else:
            flash("Invalid email or password!", "danger")
            return render_template('login.html')
    return render_template('login.html')

In [12]:
@app.route('/logout')
def logout():
    session.pop('email', None)
    flash("You have been logged out.", "info")
    return redirect(url_for('login'))

In [ ]:
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.27.65.59:5000
Press CTRL+C to quit
10.27.65.136 - - [09/Jan/2026 10:51:32] "POST /transport HTTP/1.1" 200 -


Received data: {'data': '24#45#73#12.9146080#77.6361320'}
da 24#45#73#12.9146080#77.6361320
['24', '45', '73', '12.9146080', '77.6361320']


10.27.65.136 - - [09/Jan/2026 10:51:42] "POST /transport HTTP/1.1" 200 -


Received data: {'data': '24#46#73#12.9146080#77.6361320'}
da 24#46#73#12.9146080#77.6361320
['24', '46', '73', '12.9146080', '77.6361320']


...

Received data: {'data': '24#44#70#12.9146080#77.6361320'}
da 24#44#70#12.9146080#77.6361320
['24', '44', '70', '12.9146080', '77.6361320']


10.27.65.136 - - [09/Jan/2026 11:14:22] "POST /transport HTTP/1.1" 200 -


Received data: {'data': '24#44#69#12.9146080#77.6361320'}
da 24#44#69#12.9146080#77.6361320
['24', '44', '69', '12.9146080', '77.6361320']


10.27.65.136 - - [09/Jan/2026 11:14:28] "POST /transport HTTP/1.1" 200 -


Received data: {'data': '24#44#69#12.9146080#77.6361320'}
da 24#44#69#12.9146080#77.6361320
['24', '44', '69', '12.9146080', '77.6361320']
